# Person 2 — Patient-separated split and training-only preprocessing

Follows the "Next team member" instructions in `aki_handoff/README.txt`:

* Use `predictor_columns.csv` to select inputs explicitly.
* Never use identifiers, outcome fields or ascertainment fields as predictors.
* Create shared patient-separated partitions using `subject_id`.
* Fit imputation, encoding, scaling and text processing on **training data only**.

**Design (proposed in the team methodology document):**

* 70 / 15 / 15 train / validation / test, split **by patient**.
* Stratified at patient level on the worst observed outcome (none / stage 1 only / stage ≥2),
  so that both targets have similar frequencies in every partition.
* One fixed seed, run once. The seed is not changed to get a "nicer" split.
* The natural outcome frequency is kept: no resampling or rebalancing.

Outputs feed Person 3 (baselines), Person 4 (XGBoost) and Person 5 (evaluation).

In [ ]:
import sys
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import scipy.sparse as sp
import sklearn
from sklearn.model_selection import train_test_split

sys.path.insert(0, str(Path.cwd()))
import aki_preprocessing as ap

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)

candidates = [Path("../aki_handoff"), Path("aki_handoff"), Path("/content/aki_handoff")]
DATA_DIR = next(p for p in candidates if (p / "modelling_data.csv").exists())
OUT_DIR = Path("outputs")
(OUT_DIR / "preprocessors").mkdir(parents=True, exist_ok=True)
(OUT_DIR / "matrices").mkdir(parents=True, exist_ok=True)

SEED = 90089          # fixed once; do not tune
TRAIN, VAL, TEST = 0.70, 0.15, 0.15

print("Data folder:", DATA_DIR.resolve())
print("pandas", pd.__version__, "| scikit-learn", sklearn.__version__, "| numpy", np.__version__)

## 1. Load the modelling table and select inputs explicitly

In [ ]:
df = pd.read_csv(DATA_DIR / "modelling_data.csv")
predictors = pd.read_csv(DATA_DIR / "predictor_columns.csv")["predictor"].tolist()

assert predictors == ap.PREDICTORS, "predictor_columns.csv differs from aki_preprocessing.PREDICTORS"
forbidden = {"subject_id", "stay_id", "hadm_id", "aki_label", "aki_stage2_label"}
assert not forbidden & set(predictors)
assert set(ap.NUMERIC + ap.CATEGORICAL + [ap.TEXT]) == set(predictors)

# Check types after loading (CSV does not keep pandas types)
for c in ap.NUMERIC:
    assert pd.api.types.is_numeric_dtype(df[c]), c
print(df[predictors].dtypes.to_string())
print("\nRows:", len(df), "| patients:", df["subject_id"].nunique(),
      "| duplicate stay_id:", df["stay_id"].duplicated().sum())

## 2. Patient-level stratified split

Each patient gets one stratum from their worst labelled encounter. Patients, not encounters,
are split 70/30 and then 50/50, so every encounter of a patient lands in the same partition.

In [ ]:
patients = df.groupby("subject_id").agg(
    encounters=("stay_id", "size"),
    any_aki=("aki_label", "max"),
    any_stage2=("aki_stage2_label", "max"),
).reset_index()
patients["stratum"] = np.select(
    [patients["any_stage2"] == 1, patients["any_aki"] == 1], ["stage2", "stage1_only"], "none")
print(patients["stratum"].value_counts().to_string())

train_p, hold_p = train_test_split(patients, test_size=VAL + TEST,
                                   stratify=patients["stratum"], random_state=SEED)
val_p, test_p = train_test_split(hold_p, test_size=TEST / (VAL + TEST),
                                 stratify=hold_p["stratum"], random_state=SEED)

partition_of = pd.concat([
    pd.Series("train", index=train_p["subject_id"]),
    pd.Series("validation", index=val_p["subject_id"]),
    pd.Series("test", index=test_p["subject_id"]),
])
df["partition"] = df["subject_id"].map(partition_of)

manifest = df[["subject_id", "stay_id", "partition"]].sort_values(["partition", "subject_id", "stay_id"])
manifest.to_csv(OUT_DIR / "split_manifest.csv", index=False)
print("\nManifest rows:", len(manifest))

## 3. Split checks

In [ ]:
parts = ["train", "validation", "test"]
sets = {p: set(df.loc[df["partition"] == p, "subject_id"]) for p in parts}
checks = {
    "Every encounter assigned": int(df["partition"].isna().sum()) == 0,
    "Each stay_id appears once": not manifest["stay_id"].duplicated().any(),
    "Patients in >1 partition": max(df.groupby("subject_id")["partition"].nunique()) == 1,
    "train ∩ validation patients": len(sets["train"] & sets["validation"]) == 0,
    "train ∩ test patients": len(sets["train"] & sets["test"]) == 0,
    "validation ∩ test patients": len(sets["validation"] & sets["test"]) == 0,
    "All 113,097 encounters covered": len(manifest) == 113097,
}
for k, v in checks.items():
    print(f"{'PASS' if v else 'FAIL'}  {k}")
assert all(checks.values())

summary = []
for p in parts + ["all"]:
    d = df if p == "all" else df[df["partition"] == p]
    summary.append({
        "partition": p,
        "patients": d["subject_id"].nunique(),
        "patients %": f"{d['subject_id'].nunique() / df['subject_id'].nunique():.1%}",
        "encounters": len(d),
        "encounters %": f"{len(d) / len(df):.1%}",
        "primary positives": int(d["aki_label"].sum()),
        "primary prevalence": f"{d['aki_label'].mean():.2%}",
        "stage ≥2 positives": int(d["aki_stage2_label"].sum()),
        "stage ≥2 prevalence": f"{d['aki_stage2_label'].mean():.2%}",
        "age median": d["age"].median(),
        "all six vitals missing": f"{d[['temperature','heartrate','resprate','o2sat','sbp','dbp']].isna().all(axis=1).mean():.1%}",
        "max encounters/patient": int(d.groupby("subject_id").size().max()),
    })
split_summary = pd.DataFrame(summary)
split_summary.to_csv(OUT_DIR / "split_summary.csv", index=False, encoding="utf-8-sig")
split_summary

## 4. Partition tables (row order = matrix row order)

Each partition's identifiers and both targets are saved in the same row order as its
feature matrices. `X_<partition>.csv` holds the **unprocessed** cleaned predictors, for
anyone who prefers to fit `aki_preprocessing` themselves (for example inside
cross-validation folds).

In [ ]:
data = {p: df[df["partition"] == p].sort_values("stay_id").reset_index(drop=True) for p in parts}
for p, d in data.items():
    d[["subject_id", "stay_id", "partition", "aki_label", "aki_stage2_label"]].to_csv(
        OUT_DIR / "matrices" / f"labels_{p}.csv", index=False)
    d[["stay_id"] + predictors].to_csv(OUT_DIR / "matrices" / f"X_raw_{p}.csv", index=False)
    print(p, d.shape)

X = {p: d[predictors] for p, d in data.items()}

## 5. Fit preprocessing on the training partition only

| Variant | Numeric (age, 6 vitals, acuity, pain score) | Missingness | Categorical (gender, transport, pain status) | Chief complaint |
|---|---|---|---|---|
| `lr_structured` | training median → standardised | 0/1 indicator for each of 8 fields | one-hot, unseen → all zeros | – |
| `lr_text` | same | same | same | TF-IDF (1–2 grams, min_df 5, sublinear tf), vocabulary from training only |
| `tree_structured` | passed through, **NaN kept** | implicit (NaN) | one-hot | – |
| `tree_text` | same | same | same | TF-IDF as above |

* Acuity is treated as ordinal numeric; rare category counts are suppressed in the public copy.
* Missingness indicators are included because Person 2's descriptive analysis found that
  missing vital signs are strongly associated with AKI (13.5% vs 6.4%).
* The tokeniser keeps clinical shorthand such as `s/p`, `n/v` and `r/o` as single tokens, and
  text is lowercased (the data mixes "Abd pain" and "ABD PAIN").

In [ ]:
fitted, feature_names, matrices = {}, {}, {}
for name, builder in ap.PREPROCESSORS.items():
    pre = builder()
    pre.fit(X["train"])                                    # TRAINING ONLY
    fitted[name] = pre
    feature_names[name] = list(pre.get_feature_names_out())
    matrices[name] = {p: pre.transform(X[p]) for p in parts}   # transform only
    shapes = {p: matrices[name][p].shape for p in parts}
    kind = "sparse" if sp.issparse(matrices[name]["train"]) else "dense"
    print(f"{name:16s} {kind:6s} features={len(feature_names[name]):5d}  {shapes}")

## 6. Leakage and behaviour checks

In [ ]:
leak = []

# 6a. Imputation medians equal the TRAINING medians, not the full-data medians
imp = fitted["lr_structured"].named_transformers_["num"].named_steps["impute"]
train_med = X["train"][ap.NUMERIC].median()
full_med = df[ap.NUMERIC].median()
leak.append({"check": "Imputer medians == training medians",
             "result": bool(np.allclose(imp.statistics_, train_med.values))})
med_table = pd.DataFrame({"training median (used)": train_med, "all-data median (not used)": full_med})

# 6b. Scaler fitted on training: training columns have mean 0 / sd 1; validation does not exactly
scaler = fitted["lr_structured"].named_transformers_["num"].named_steps["scale"]
n_num = len(ap.NUMERIC)
tr_num = matrices["lr_structured"]["train"][:, :n_num]
va_num = matrices["lr_structured"]["validation"][:, :n_num]
leak.append({"check": "Scaled training numeric means ≈ 0",
             "result": bool(np.allclose(tr_num.mean(axis=0), 0, atol=1e-8))})
leak.append({"check": "Scaler mean_ == training means after imputation",
             "result": bool(np.allclose(scaler.mean_, X["train"][ap.NUMERIC].fillna(train_med).mean().values))})
leak.append({"check": "Validation scaled means are not forced to 0 (transform only)",
             "result": bool(not np.allclose(va_num.mean(axis=0), 0, atol=1e-8))})

# 6c. TF-IDF vocabulary built from training text only
vec = fitted["lr_text"].named_transformers_["text"].named_steps["tfidf"]
from sklearn.feature_extraction.text import TfidfVectorizer
ref = TfidfVectorizer(token_pattern=ap.TOKEN_PATTERN, ngram_range=(1, 2), min_df=5, sublinear_tf=True)
ref.fit(ap.clean_text(X["train"][ap.TEXT]))
leak.append({"check": "TF-IDF vocabulary == vocabulary refit on training text",
             "result": vec.vocabulary_ == ref.vocabulary_})
ref_all = TfidfVectorizer(token_pattern=ap.TOKEN_PATTERN, ngram_range=(1, 2), min_df=5, sublinear_tf=True)
ref_all.fit(ap.clean_text(df[ap.TEXT]))
leak.append({"check": "TF-IDF vocabulary differs from an all-data fit (so a leak would show)",
             "result": vec.vocabulary_ != ref_all.vocabulary_})

# 6d. Unseen categories are handled (all-zero one-hot), not an error
probe = X["validation"].head(1).copy()
probe["arrival_transport"] = "NEVER_SEEN"
probe["gender"] = "X"
out = fitted["lr_structured"].transform(probe)
cat_idx = [i for i, f in enumerate(feature_names["lr_structured"]) if f.startswith("cat__arrival_transport") or f.startswith("cat__gender")]
leak.append({"check": "Unseen category encodes to all zeros", "result": bool(np.all(out[0, cat_idx] == 0))})

# 6e. No identifier or outcome sneaks into features
bad = [f for name in feature_names for f in feature_names[name]
       if any(k in f for k in ["subject_id", "stay_id", "hadm_id", "aki_label", "stage2"])]
leak.append({"check": "No identifier/outcome in any feature name", "result": len(bad) == 0})

# 6f. LR matrices have no NaN; tree matrices keep NaN
leak.append({"check": "No NaN in LR matrices",
             "result": all(not np.isnan(matrices[n][p].toarray() if sp.issparse(matrices[n][p]) else matrices[n][p]).any()
                           for n in ["lr_structured", "lr_text"] for p in parts)})
leak.append({"check": "Tree structured matrix keeps NaN for missing vitals",
             "result": int(np.isnan(matrices["tree_structured"]["train"]).sum())
                       == int(X["train"][ap.NUMERIC].isna().sum().sum())})

leak = pd.DataFrame(leak)
leak.to_csv(OUT_DIR / "preprocessing_checks.csv", index=False, encoding="utf-8-sig")
assert leak["result"].all()
print(leak.to_string(index=False))
med_table

In [ ]:
# Text representation summary
vocab = vec.vocabulary_
val_tokens = set(t for s in ap.clean_text(X["validation"][ap.TEXT])
                 for t in __import__("re").findall(ap.TOKEN_PATTERN, s))
print("TF-IDF features (from training):", len(vocab))
print("Validation unigrams not in the training vocabulary:", len(val_tokens - set(vocab)),
      "of", len(val_tokens), "(ignored at transform time, as intended)")
print("Training rows with empty text after cleaning:", int((ap.clean_text(X['train'][ap.TEXT]) == "").sum()))
idf = pd.Series(vec.idf_, index=vec.get_feature_names_out()).sort_values()
print("\nMost common training terms (lowest IDF):")
print(idf.head(25).round(2).to_string())

## 7. Save fitted preprocessors, feature names and matrices

* `preprocessors/<variant>.joblib` contains the fitted ColumnTransformers. Loading them needs
  `aki_preprocessing.py` on the Python path and a compatible scikit-learn version (the one used
  is recorded in `environment.txt`). If versions differ, refit from `X_raw_train.csv` with
  `aki_preprocessing.PREPROCESSORS[variant]()` instead.
* Dense matrices are saved as `.npy` and sparse ones (text variants) as `.npz`.
* **XGBoost caution:** when given a sparse CSR matrix, XGBoost treats *absent* entries as
  missing, not as 0. For `tree_structured`, use the dense `.npy` (NaN = missing). For
  `tree_text`, that behaviour also applies to the one-hot and TF-IDF zeros. This is usually
  acceptable for trees, but Person 4 should know about it.

In [ ]:
for name in fitted:
    joblib.dump(fitted[name], OUT_DIR / "preprocessors" / f"{name}.joblib")
    pd.Series(feature_names[name], name="feature").to_csv(
        OUT_DIR / "preprocessors" / f"{name}_feature_names.csv", index=False)
    for p in parts:
        m = matrices[name][p]
        if sp.issparse(m):
            sp.save_npz(OUT_DIR / "matrices" / f"{name}_{p}.npz", m.tocsr())
        else:
            np.save(OUT_DIR / "matrices" / f"{name}_{p}.npy", m.astype(np.float64))

(OUT_DIR / "environment.txt").write_text(
    f"python {sys.version.split()[0]}\npandas {pd.__version__}\nnumpy {np.__version__}\n"
    f"scikit-learn {sklearn.__version__}\nscipy {sp.__name__ and __import__('scipy').__version__}\n"
    f"joblib {joblib.__version__}\nsplit seed {SEED}\n", encoding="utf-8")

# Round trip: reload one preprocessor and confirm identical output
reloaded = joblib.load(OUT_DIR / "preprocessors" / "lr_text.joblib")
same = (reloaded.transform(X["test"]) != matrices["lr_text"]["test"]).nnz == 0
print("Reloaded lr_text reproduces test matrix:", same)
assert same

## 8. How to use (for Persons 3–5)

```python
import numpy as np, pandas as pd, scipy.sparse as sp, joblib, sys
sys.path.insert(0, "person2_split_preprocessing")
M = "person2_split_preprocessing/outputs/matrices"

X_tr = np.load(f"{M}/lr_structured_train.npy")          # or sp.load_npz(f"{M}/lr_text_train.npz")
y_tr = pd.read_csv(f"{M}/labels_train.csv")["aki_label"]  # or "aki_stage2_label"
X_va = np.load(f"{M}/lr_structured_validation.npy")
ids_va = pd.read_csv(f"{M}/labels_validation.csv")        # subject_id, stay_id for prediction files
```

* **Cross-validation within training:** do not reuse these fitted preprocessors inside the
  folds. Rebuild with `aki_preprocessing.PREPROCESSORS[variant]()` in a `Pipeline` with the
  model, and use `GroupKFold` / `StratifiedGroupKFold` on `subject_id`. Preprocessing is then
  refit within each fold.
* **Test partition:** do not touch it until the team records that models are frozen.
* **Prediction file format (team agreement):** `stay_id, subject_id, target, observed label,
  predicted probability, model name, partition`.

In [ ]:
for f in sorted(OUT_DIR.rglob("*")):
    if f.is_file():
        print(f"{str(f.relative_to(OUT_DIR)):55s} {f.stat().st_size / 1024:9.1f} KB")